Blender Python API features:

* Edit any data the user interface can (Scenes, Meshes, Particles etc.).
* Modify user preferences, keymaps and themes.
* Run tools with own settings.
* Create user interface elements such as menus, headers and panels.
* Create new tools.
* Create interactive tools.
* Create new rendering engines that integrate with Blender.
* Subscribe to changes to data and it’s properties.
* Define new settings in existing Blender data.
* Draw in the 3D Viewport using Python.

(Still) missing features:

* Create new space types.
* Assign custom properties to every type.

## Before Starting

This document is intended to familiarize you with Blender Python API but not to fully cover each topic.

A quick list of helpful things to know before starting:

* Enable [Developer Extra](https://docs.blender.org/manual/en/dev/editors/preferences/interface.html#bpy-types-preferencesview-show-developer-ui "(in Blender 5.0 Manual v5.0)") and [Python Tooltips](https://docs.blender.org/manual/en/dev/editors/preferences/interface.html#bpy-types-preferencesview-show-tooltips-python "(in Blender 5.0 Manual v5.0)").
* The [Python Console](https://docs.blender.org/manual/en/dev/editors/python_console.html#bpy-types-spaceconsole "(in Blender 5.0 Manual v5.0)") is great for testing one-liners; it has autocompletion so you can inspect the API quickly.
* Button tooltips show Python attributes and operator names (when enabled see above).
* The context menu of buttons directly links to this API documentation (when enabled see above).
* Many python examples can be found in the text editor’s template menu.
* To examine further scripts distributed with Blender, see:

  * `<span class="pre">scripts/startup/bl_ui</span>` for the user interface.
  * `<span class="pre">scripts/startup/bl_operators</span>` for operators.

  Exact location depends on platform, see: [directory layout docs](https://docs.blender.org/manual/en/dev/advanced/blender_directory_layout.html#blender-directory-layout "(in Blender 5.0 Manual v5.0)").

### Running Scripts

The two most common ways to execute Python scripts are using the built-in text editor or entering commands in the Python console. Both the *Text Editor* and *Python Console* are space types you can select from the header. Rather than manually configuring your spaces for Python development, you can use the *Scripting* workspace accessible from the Topbar tabs.

From the text editor you can open `<span class="pre">.py</span>` files or paste them from the clipboard, then test using  *Run Script* . The Python Console is typically used for typing in snippets and for testing to get immediate feedback, but can also have entire scripts pasted into it. Scripts can also run from the command line with Blender but to learn scripting in Blender this isn't essential.


## Key Concepts

### Data Access

#### Accessing Data-Blocks

You can access Blender’s data with the Python API in the same way as the animation system or user interface; this implies that any setting that can be changed via a button can also be changed with Python. Accessing data from the currently loaded blend-file is done with the module [`bpy.data`](https://docs.blender.org/api/current/bpy.data.html#module-bpy.data "bpy.data"). It gives access to library data, for example:

In [1]:
import bpy
import os

In [2]:
# Print all objects.
for obj in bpy.data.objects:
    print(obj.name)

# Print all scene names in a list.
print(bpy.data.scenes.keys())

# Remove mesh Cube.
# if "Cube" in bpy.data.meshes:
#     mesh = bpy.data.meshes["Cube"]
#     print("removing mesh", mesh)
#     bpy.data.meshes.remove(mesh)

# Write images into a file next to the blend.
with open(os.path.splitext(bpy.data.filepath)[0] + ".txt", 'w') as fs:
    for image in bpy.data.images:
        fs.write("{:s} {:d} x {:d}\n".format(image.filepath, image.size[0], image.size[1]))

Camera
Cube
Light
['Scene']


#### Accessing Collections
You will notice that an index as well as a string can be used to access members of the collection. Unlike Python dictionaries, both methods are available; however, the index of a member may change while running Blender.

In [3]:
for i in bpy.data.objects:
    print(i)

<bpy_struct, Object("Camera") at 0x000002BB7DAF7330>
<bpy_struct, Object("Cube") at 0x000002BB2919FC60>
<bpy_struct, Object("Light") at 0x000002BB291A0130>


In [4]:
print(list(bpy.data.objects))
print(bpy.data.objects["Cube"])
print(bpy.data.objects[0])

[bpy.data.objects['Camera'], bpy.data.objects['Cube'], bpy.data.objects['Light']]
<bpy_struct, Object("Cube") at 0x000002BB2919FC60>
<bpy_struct, Object("Camera") at 0x000002BB7DAF7330>


#### Accessing Attributes
Once you have a data-block, such as a material, object, collection, etc., its attributes can be accessed much like you would change a setting using the graphical interface. In fact, the tooltip for each button also displays the Python attribute which can help in finding what settings to change in a script.

In [5]:
print(bpy.data.objects[0].name)
print(bpy.data.scenes["Scene"])
print(bpy.data.materials.new("MyMaterial"))

Camera
<bpy_struct, Scene("Scene") at 0x000002BB29040180>
<bpy_struct, Material("MyMaterial") at 0x000002BB2A67D298>


For testing what data to access it’s useful to use the Python Console, which is its own space type. This supports auto-complete, giving you a fast way to explore the data in your file.

Example of a data path that can be quickly found via the console:

In [9]:
print(bpy.data.scenes[0].render.resolution_percentage)
# print(bpy.data.scenes[0].objects["Torus"].data.vertices[0].co.x)

100


#### Data Creation/Removal
When you are familiar with other Python APIs you may be surprised that new data-blocks in the bpy API cannot be created by calling the class:

In [ ]:
# bpy.types.Mesh()
# Traceback (most recent call last):
#   File "<blender_console>", line 1, in <module>
# TypeError: bpy_struct.__new__(type): expected a single argument

This is an intentional part of the API design. The Blender Python API can’t create Blender data that exists outside the main Blender database (accessed through [`bpy.data`](bpy.data.html#module-bpy.data "bpy.data")), because this data is managed by Blender (save, load, undo, append, etc).

Data is added and removed via methods on the collections in [`bpy.data`](bpy.data.html#module-bpy.data "bpy.data"), e.g:

In [10]:
mesh = bpy.data.meshes.new(name="MyMesh")
print(mesh)

<bpy_struct, Mesh("MyMesh") at 0x000002BB1048A048>


In [11]:
bpy.data.meshes.remove(mesh)

#### Custom Properties

Python can access properties on any data-block that has an ID (data that can be linked in and accessed from [`bpy.data`](bpy.data.html#module-bpy.data "bpy.data")). When assigning a property, you can pick your own names, these will be created when needed or overwritten if they already exist.

This data is saved with the blend-file and copied with objects, for example:

In [12]:
bpy.context.object["MyOwnProperty"] = 42

if "SomeProp" in bpy.context.object:
    print("Property found")

# Use the get function like a Python dictionary
# which can have a fallback value.
value = bpy.data.scenes["Scene"].get("test_prop", "fallback value")

# Dictionaries can be assigned as long as they only use basic types.
collection = bpy.data.collections.new("MyTestCollection")
collection["MySettings"] = {"foo": 10, "bar": "spam", "baz": {}}

del collection["MySettings"]

Note that these properties can only be assigned basic Python types:

* int, float, string
* array of ints or floats
* dictionary (only string keys are supported, values must be basic types too)

These properties are valid outside of Python. They can be animated by curves or used in driver paths.

### Context

While it’s useful to be able to access data directly by name or as a list, it’s more common to operate on the user’s selection. The context is always available from `bpy.context` and can be used to get the active object, scene, tool settings along with many other attributes.

Some common use cases are:

In [13]:
print(bpy.context.object)
print(bpy.context.selected_objects)
print(bpy.context.visible_bones)

<bpy_struct, Object("Cube") at 0x000002BB2919FC60>
[bpy.data.objects['Cube']]
None


Note that the context is read-only, which means that these values cannot be modified directly. But they can be changed by running API functions or by using the data API.

So `bpy.context.active_object = obj` will raise an error. But `bpy.context.view_layer.objects.active = obj` works as expected.

The context attributes change depending on where they are accessed. The 3D Viewport has different context members than the Python Console, so take care when accessing context attributes that the user state is known.

See [`bpy.context`](bpy.context.html#module-bpy.context "bpy.context") API reference.

### Operators (Tools)

Operators are tools generally accessed by the user from buttons, menu items or key shortcuts. From the user perspective they are a tool but Python can run these with its own settings through the [`bpy.ops`](bpy.ops.html#module-bpy.ops "bpy.ops") module.

Examples:

In [16]:
print(bpy.ops.mesh.flip_normals())
print(bpy.ops.mesh.hide(unselected=False))
print(bpy.ops.object.transform_apply())

RuntimeError: Operator bpy.ops.mesh.flip_normals.poll() failed, context is incorrect

> The [Operator Cheat Sheet](https://docs.blender.org/manual/en/dev/advanced/operators.html#bpy-ops-wm-operator-cheat-sheet "(in Blender 5.0 Manual v5.0)") gives a list of all operators and their default values in Python syntax, along with the generated docs. This is a good way to get an overview of all Blender’s operators.

#### Operator Poll()

Many operators have a “poll” function which checks if the cursor is in a valid area or if the object is in the correct mode (Edit Mode, Weight Paint Mode, etc). When an operator’s poll function fails within Python, an exception is raised.

For example, calling `bpy.ops.view3d.render_border()` from the console raises the following error:

```
RuntimeError: Operator bpy.ops.view3d.render_border.poll() failed, context is incorrect
```

In this case the context must be the 3D Viewport with an active camera.

To avoid using try-except clauses wherever operators are called, you can call the operators own `poll()` function to check if it can run the operator in the current context.

In [19]:
print(bpy.ops.view3d.render_border.poll())
if bpy.ops.view3d.render_border.poll():
    bpy.ops.view3d.render_border()

False


## Integration

Python scripts can integrate with Blender in the following ways:

* By defining a render engine.
* By defining operators.
* By defining menus, headers and panels.
* By inserting new buttons into existing menus, headers and panels.

In Python, this is done by defining a class, which is a subclass of an existing type.

### Example Operator

In [20]:
def main(context):
    for ob in context.scene.objects:
        print(ob)

class SimpleOperator(bpy.types.Operator):
    """Tooltip"""
    bl_idname = "object.simple_operator"
    bl_label = "Simple Object Operator"

    @classmethod
    def poll(cls, context):
        return context.active_object is not None

    def execute(self, context):
        main(context)
        return {'FINISHED'}


def menu_func(self, context):
    self.layout.operator(SimpleOperator.bl_idname, text=SimpleOperator.bl_label)


# Register and add to the "object" menu (required to also use F3 search "Simple Object Operator" for quick access).
def register():
    bpy.utils.register_class(SimpleOperator)
    bpy.types.VIEW3D_MT_object.append(menu_func)


def unregister():
    bpy.utils.unregister_class(SimpleOperator)
    bpy.types.VIEW3D_MT_object.remove(menu_func)


if __name__ == "__main__":
    register()
    # Test call.
    bpy.ops.object.simple_operator()

<bpy_struct, Object("Cube") at 0x000002BB2919FC60>
<bpy_struct, Object("Light") at 0x000002BB291A0130>
<bpy_struct, Object("Camera") at 0x000002BB7DAF7330>


Once this script runs, `SimpleOperator` is registered with Blender and can be called from Operator Search or added to the toolbar.

To run the script:

1. Start Blender and switch to the Scripting workspace.
2. Click the *New* button in the text editor to create a new text data-block.
3. Copy the code from above and paste it into the text editor.
4. Click on the *Run Script* button.
5. Move your cursor into the 3D Viewport, open the [Operator Search menu](https://docs.blender.org/manual/en/dev/interface/operators.html#bpy-ops-wm-search-menu "(in Blender 5.0 Manual v5.0)"), and type “Simple”.
6. Click on the “Simple Operator” item found in search.

> The class members with the `bl_` prefix are documented in the API reference [`bpy.types.Operator`](bpy.types.Operator.html#bpy.types.Operator "bpy.types.Operator").


> The output from the `main` function is sent to the terminal; in order to see this, be sure to [use the terminal](info_tips_and_tricks.html#use-the-terminal).